# 03주차 · 분포 가설과 단어 임베딩

**임베딩 기반 데이터 과학 — 국립목포대학교 컴퓨터학부 4학년**

이 Notebook은 *Hands-On Large Language Models* 공개 저장소의 개념과 실습 흐름을
한국어 수업에 맞게 새로 구성한 파생 강의자료입니다. 원본은 Apache License 2.0을
따르며, 출처와 변경 사항은 `SOURCE_AND_LICENSE.md`에 기록했습니다.

- 원본: https://github.com/HandsOnLLM/Hands-On-Large-Language-Models
- 기준 커밋: `ea3390819997999a51983677b80b3aac4dc50ada`
- 권장 환경: Google Colab 또는 Python 3.11+


## 학습목표

- 분포 가설과 단어 동시 출현 행렬을 설명한다.
- PPMI가 흔한 문맥의 영향을 줄이는 원리를 이해한다.
- SVD로 저차원 단어 임베딩을 구성한다.


In [ ]:
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 100)

def cosine(a, b):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(a @ b / denom) if denom else 0.0


In [ ]:
corpus = [
    "목포 바다 관광 여행",
    "여수 바다 관광 여행",
    "목포 해양 관광 산업",
    "조선 해양 제조 산업",
    "배터리 제조 공정 산업",
    "배터리 소재 제조 기술",
]
tokens = [s.split() for s in corpus]
vocab = sorted({w for sent in tokens for w in sent})
index = {w: i for i, w in enumerate(vocab)}
vocab


In [ ]:
window = 2
C = np.zeros((len(vocab), len(vocab)))
for sent in tokens:
    for i, word in enumerate(sent):
        for j in range(max(0, i-window), min(len(sent), i+window+1)):
            if i != j:
                C[index[word], index[sent[j]]] += 1
pd.DataFrame(C, index=vocab, columns=vocab)


## PPMI

$$PMI(w,c)=\log\frac{P(w,c)}{P(w)P(c)},\quad PPMI=\max(PMI,0)$$


In [ ]:
total = C.sum()
p_wc = C / total
p_w = C.sum(axis=1, keepdims=True) / total
p_c = C.sum(axis=0, keepdims=True) / total
with np.errstate(divide="ignore", invalid="ignore"):
    ppmi = np.maximum(np.log2(p_wc / (p_w @ p_c)), 0)
ppmi[~np.isfinite(ppmi)] = 0
U, S, VT = np.linalg.svd(ppmi, full_matrices=False)
E = U[:, :2] * np.sqrt(S[:2])
emb = {w: E[index[w]] for w in vocab}


In [ ]:
target = "목포"
ranking = sorted(((w, cosine(emb[target], emb[w])) for w in vocab if w != target), key=lambda x: -x[1])
ranking[:5]


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(E[:, 0], E[:, 1])
for w, v in emb.items(): ax.annotate(w, v + 0.02)
ax.set_title("PPMI-SVD 단어 임베딩")
plt.show()


## 학생 활동

- 말뭉치에 문장 10개를 추가하고 결과 변화를 기록하라.
- 윈도 크기 1과 3을 비교하라.
- 작은 말뭉치에서 유사도 결과가 불안정한 이유를 설명하라.


---
## 학습 기록과 생성형 AI 사용 내역

다음 항목을 자신의 말로 작성하세요.

1. 이번 실습에서 가장 중요한 결과는 무엇인가?
2. 결과를 뒷받침하는 수치 또는 그래프는 무엇인가?
3. 실패하거나 예상과 달랐던 부분은 무엇인가?
4. 생성형 AI를 사용했다면 프롬프트, 채택·거부한 제안, 직접 검증한 내용을 기록하라.
